# Description

In this notebook, I will train the Tokenizer (SentencePiece for T5 model) from the dataset

In [19]:
import io
import numpy as np
import pandas as pd
import os
import tempfile
from pathlib import Path
from typing import List, Optional

import sentencepiece as spm
from transformers import T5Tokenizer, T5TokenizerFast

In [20]:
PATH_FILE_DATA = os.path.join(os.getcwd(), 'dataset', 'processed', 'processed_data.csv')

# 1. Read data

In [21]:
df = pd.read_csv(PATH_FILE_DATA)
print(f"Number of records in processed data: {len(df)}")

Number of records in processed data: 1371223


In [22]:
list_python_function = df['method_code'].tolist()
print(f"Number of python functions: {len(list_python_function)}")

Number of python functions: 1371223


In [23]:
idx = np.random.randint(0, len(list_python_function))
print("Example of python function:")
print(list_python_function[idx])

Example of python function:
def test_listen_for_funding_info_update_updates_funding_info(self, mock_api, mock_queue_get):
        rate_regex_url = re.compile(
            f"^{web_utils.get_rest_url_for_endpoint(CONSTANTS.GET_LAST_FUNDING_RATE_PATH_URL)}".replace(".",
                                                                                                        r"\.").replace(
                "?", r"\?")
        )
        interest_regex_url = re.compile(
            f"^{web_utils.get_rest_url_for_endpoint(CONSTANTS.OPEN_INTEREST_PATH_URL)}".replace(".", r"\.").replace("?",
                                                                                                                    r"\?")
        )
        mark_regex_url = re.compile(
            f"^{web_utils.get_rest_url_for_endpoint(CONSTANTS.MARK_PRICE_PATH_URL)}".replace(".", r"\.").replace("?",
                                                                                                                 r"\?")
    

In [ ]:
def write_corpus_txt_from_list(codes: List[str], path: str):
    with io.open(path, "w", encoding="utf-8") as f:
        for s in codes:
            s = s.replace("\r\n", "\n").strip()
            if not s:
                continue
            f.write(" ".join(s.splitlines()) + "\n")
        

In [ ]:
def train_sentencepiece(
    input_path: str,
    model_prefix: str,
    vocab_size: int = 32000,
    model_type: str = "unigram",
    character_coverage: float = 1.0,
    input_sentence_size: int = 1_000_000,
    shuffle_input_sentence: bool = True,
):
    spm.SentencePieceTrainer.Train(
        input=input_path,
        model_prefix=model_prefix,
        vocab_size=vocab_size,
        model_type=model_type,
        character_coverage=character_coverage,
        input_sentence_size=input_sentence_size,
        shuffle_input_sentence=shuffle_input_sentence,
    )

def wrap_as_hf_tokenizer(spm_model_path: str, out_dir: str, use_fast: bool = False, extra_ids: int = 100):
    """
    Wrap the SentencePiece model as a HF T5 tokenizer.
    `extra_ids` creates <extra_id_0> .. <extra_id_{N-1}> for span infilling.
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    if use_fast:
        tok = T5TokenizerFast(vocab_file=str(spm_model_path), extra_ids=extra_ids)
    else:
        tok = T5Tokenizer(vocab_file=str(spm_model_path), extra_ids=extra_ids)


    if tok.pad_token is None:
        tok.add_special_tokens({"pad_token": "<pad>"})

    tok.save_pretrained(str(out_dir))
    print(f"[Done] Saved HF tokenizer to {out_dir}")

In [ ]:
def main(
    out_dir: str,
    vocab_size: int,
    CODES: Optional[List[str]] = None,
    use_fast: bool = False,
):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    with tempfile.TemporaryDirectory() as tmpd:
        corpus_path = Path(tmpd) / "corpus.txt"

        if not CODES:
            raise ValueError("No input provided. Pass --input_files or fill CODES list in the script.")
        write_corpus_txt_from_list(CODES, str(corpus_path))

        model_prefix = str(Path(tmpd) / "t5_code_spm")
        print(f"[INFO] Training SentencePiece (vocab_size={vocab_size})...")
        train_sentencepiece(
            input_path=str(corpus_path),
            model_prefix=model_prefix,
            vocab_size=vocab_size,
        )
        spm_model = f"{model_prefix}.model"
        print(f"[DONE] Trained SPM model: {spm_model}")

        wrap_as_hf_tokenizer(spm_model_path=spm_model, out_dir=str(out_dir), use_fast=use_fast)

In [28]:
main(out_dir="./tokenizer_t5_code",
    vocab_size=32000,
    CODES=list_python_function,
    use_fast=False)
print("[DONE] Tokenizer training complete.")

sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: /tmp/tmpy8xrxyrb/corpus.txt
  input_format: 
  model_prefix: /tmp/tmpy8xrxyrb/t5_code_spm
  model_type: UNIGRAM
  vocab_size: 32000
  self_test_sample_size: 0
  character_coverage: 1
  input_sentence_size: 20000000
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 0
  bos_id: 1
  eos_id: 2
  pad_id: -1
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  

[INFO] Training SentencePiece (vocab_size=32000)...
[OK] Trained SPM model: /tmp/tmpy8xrxyrb/t5_code_spm.model
[OK] Saved HF tokenizer to tokenizer_t5_code
[DONE] Tokenizer training complete.


# Check again

In [29]:
from transformers import T5Tokenizer

tok = T5Tokenizer.from_pretrained("tokenizer_t5_code") 
print(tok.pad_token, tok.pad_token_id)  # '<pad>', usually 0
ids = tok.encode("predict_if_condition: if <extra_id_0>:", add_special_tokens=True)
print(ids[:20])
print(tok.decode(ids))


<pad> 32100
[2818, 3, 905, 3, 2203, 9, 18, 32099, 92, 2]
predict_if_condition: if <extra_id_0> :</s>
